# Quadruped Fold — MJX GPU Training

## 1. GPU & XLA

In [ ]:
import os, subprocess
r = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
    shell=True, capture_output=True, text=True)
print(f"GPU: {r.stdout.strip()}")
os.environ.setdefault('XLA_FLAGS','')
if 'triton_gemm_any' not in os.environ['XLA_FLAGS']:
    os.environ['XLA_FLAGS'] += ' --xla_gpu_triton_gemm_any=True'
import jax, jax.numpy as jp
jax.config.update('jax_default_matmul_precision', 'high')
print(f"JAX {jax.__version__}  devices={jax.device_count()}")

## 2. Load Model

In [ ]:
import mujoco, numpy as np
from brax.io import mjcf
from etils import epath

XML = "test_mujoco/quadruped.xml"
sys_brax = mjcf.load(str(epath.Path(XML)))
m = mujoco.MjModel.from_xml_path(XML)

JOINT_NAMES = [
    "torso_hinge",
    "fl_hip_abd","fl_hip_flex","fl_knee",
    "fr_hip_abd","fr_hip_flex","fr_knee",
    "hl_hip_abd","hl_hip_flex","hl_knee",
    "hr_hip_abd","hr_hip_flex","hr_knee",
]
N = len(JOINT_NAMES)

# actuator 顺序验证
for i in range(m.nu):
    a = mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    assert a.replace('_act','') == JOINT_NAMES[i], f"idx {i}: {a} != {JOINT_NAMES[i]}"

# 关节 qpos 地址
JNT_ADR = np.array([m.jnt_qposadr[mujoco.mj_name2id(m,mujoco.mjtObj.mjOBJ_JOINT,n)] for n in JOINT_NAMES])

# 找 torso_front 在 Brax body 数组中的索引
torso_mj_id = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, "torso_front")
TORSO_IDX = 0
for i in range(sys_brax.num_bodies()):
    if sys_brax.body_id[i] == torso_mj_id:
        TORSO_IDX = i
        break

print(f"Brax: q={sys_brax.q_size()} v={sys_brax.qd_size()} u={sys_brax.actuator_size()}")
print(f"MuJoCo: nq={m.nq} nv={m.nv} nu={m.nu}")
print(f"torso Brax idx = {TORSO_IDX} (MuJoCo id={torso_mj_id})")
print(f"Joint qpos addr: {JNT_ADR}")

## 3. Target Poses

In [ ]:
STAND = jp.array([0.0, 0.0,-0.4,0.2, 0.0,-0.4,0.2, 0.0,0.4,-0.2, 0.0,0.4,-0.2])
FOLD  = jp.array([-1.0, 0.0,-1.0,1.0, 0.0,-1.0,1.0, 0.0,1.0,-1.0, 0.0,1.0,-1.0])

## 4. Brax Environment

In [ ]:
from brax.envs.base import PipelineEnv, State
from flax import struct

@struct.dataclass
class FoldInfo:
    init_err: jp.ndarray
    best_err: jp.ndarray

class FoldEnv(PipelineEnv):
    """obs(38)=q(13)+qd(13)+rot(9)+vel(3)  act(13)=target joint angle"""

    def __init__(self, sys_brax, stand, fold, jnt, torso_id, action_repeat=20):
        super().__init__(sys_brax, backend='mjx', n_frames=action_repeat)
        self.stand    = stand
        self.fold     = fold
        self.jnt      = jp.array(jnt)
        self.torso_id = torso_id

    @property
    def action_size(self):      return N
    @property
    def observation_size(self): return N*2 + 9 + 3

    def reset(self, rng):
        r1, r2 = jax.random.split(rng)
        noise  = jax.random.uniform(r1, (N,), minval=-0.05, maxval=0.05)
        init_q = jp.clip(self.stand + noise, -1.5, 1.5)
        init_z = 0.40 + jax.random.uniform(r2, (), minval=-0.02, maxval=0.05)
        qpos = self.sys.init_q.at[2].set(init_z).at[self.jnt].set(init_q)
        ps   = self.pipeline_init(qpos, jp.zeros(self.sys.qd_size()))
        obs  = self._obs(ps)
        err  = jp.mean(jp.abs(init_q - self.fold))
        return State(ps, obs, jp.zeros(()), jp.zeros(()), {}, FoldInfo(err, err))

    def step(self, state, action):
        ps = self.pipeline_step(state.pipeline_state, action)
        obs = self._obs(ps)
        q = ps.q[self.jnt]
        err = jp.mean(jp.abs(q - self.fold))
        zup = ps.x.rot[self.torso_id, 2, 2]
        r   = -err + (state.info.init_err - err)*3.0 - 2.0*jp.maximum(0,0.9-zup) + 0.05
        done = (zup < 0.3) | (ps.x.pos[self.torso_id, 2] < 0.05)
        return state.replace(pipeline_state=ps, obs=obs, reward=r, done=done,
                             info=FoldInfo(state.info.init_err, jp.minimum(state.info.best_err, err)))

    def _obs(self, ps):
        q   = ps.q[self.jnt]
        qd  = ps.qd[6:6+N]               # skip freejoint vel (6 DOF)
        rot = ps.x.rot[self.torso_id].ravel()  # torso rotation (3x3 → 9)
        vel = ps.xd.pos[self.torso_id]         # torso velocity (3)
        return jp.concatenate([q, qd, rot, vel])


env = FoldEnv(sys_brax, STAND, FOLD, JNT_ADR, TORSO_IDX)
print(f"obs={env.observation_size}  act={env.action_size}  torso_idx={TORSO_IDX}")

rng = jax.random.PRNGKey(0)
st  = jax.jit(env.reset)(rng)
st  = jax.jit(env.step)(st, jp.zeros(N))
print(f"smoke test OK  reward={st.reward:.3f}")

## 5. Benchmark

In [ ]:
from mujoco.mjx import benchmark
B = 4096
_, rt, steps = benchmark(mjcf.load(XML).mj_model, batch_size=B)
print(f"batch={B}  physics steps/s={steps/rt:,.0f}")

## 6. Train

In [ ]:
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks
import pickle
from datetime import datetime

def make_env():
    return FoldEnv(sys_brax, STAND, FOLD, JNT_ADR, TORSO_IDX)

train_fn, _, _ = ppo(
    environment=make_env,
    num_timesteps=50_000_000,
    episode_length=250,
    num_envs=4096,
    learning_rate=3e-4,
    entropy_cost=1e-2,
    discounting=0.97,
    unroll_length=20,
    num_minibatches=32,
    num_updates_per_batch=4,
    batch_size=256,
    reward_scaling=1.0,
    normalize_observations=True,
    network_factory=networks.make_ppo_networks,
    progress_fn=lambda s,m: print(f"\rstep {s:>10,}  rew={m.get('eval/episode_reward',0):.2f}", end=''),
    logging=True, wandb_logging=False,
)
print("\ndone")
with open(f"fold_{datetime.now():%m%d_%H%M}.pkl","wb") as f:
    pickle.dump(train_fn, f)
print("saved")